# Boundaries, and the region we never grew

`output7.ipynb` got their stated &ge;10/&ge;10 rule to **0.913** on the filtered file, with the whole
residual in one column: **users 0.76&times;** (5,733 against their 7,507) and its mirror,
check-ins per user 1.37&times;.

**The boundary question, answered by direction.** Every stricter reading of the filter moves
users *down*, and we need them up 31%:

| choice | effect on `#users` |
|---|---|
| `>=10` vs `>10` | `>10` drops users sitting exactly at 10 &rarr; **fewer** |
| "interactions" as check-ins vs distinct POIs | distinct is stricter &rarr; **fewer** |
| POI filter before user filter | rare-POI removal pushes users under 10 &rarr; **fewer** |
| iterate to a fixed point | feedback removes more &rarr; **fewer** |
| `km <= R` vs `km < R` | a handful of venues exactly at R &rarr; **~0** |
| **region larger than our boxes** | more filtered users fall in-region &rarr; **more** |

Users sitting exactly at the threshold are a few hundred out of 14,401 &mdash; low single digits
against a 1,774-user gap. So boundaries are worth *checking*, but they are not the explanation.
They are swept here anyway, cheaply, because once the other columns land they matter at the
margin (categories are still 0.97&times;).

**The axis that can close it, and has never been tested.** Every region sweep so far ran
*inward* &mdash; a shrinking radius from a fixed bounding box. We never grew the box. Ours was
validated against Section 3's NYC POI count, never against how many filtered-file users a metro
should hold.

And growing it forces a distinction the pipeline already makes. A wider region raises users
(wanted) but also check-ins (already 1.04&times;) and POIs (already 0.96&times;) &mdash; so the **collection
region and the catalogue region cannot be the same object**. That is exactly how [28] builds it:
candidates come from k-means clusters inside a city, which is not the data footprint. This
notebook therefore parameterises the two regions **separately**.

Needs `raw_POIs.txt` as well, since a wider box contains venues our current POI table does not.

## 0. Setup and the padded venue universe

In [ ]:
import os, re, gc, math, zipfile, subprocess, itertools
import pandas as pd, numpy as np

WORK = "/kaggle/working"; os.makedirs(WORK, exist_ok=True)
CHUNK = 2_000_000
TARGET = dict(users=7_507, pois=80_962, cats=436, ck=1_214_631)
CITY_CENTRE = {"New York": (40.707864, -73.905237), "Chicago": (41.826546, -87.641298),
               "Los Angeles": (34.000002, -118.250001)}
BBOX = {"New York": (-74.3, -73.6, 40.4, 41.0), "Chicago": (-88.0, -87.5, 41.6, 42.1),
        "Los Angeles": (-118.7, -117.6, 33.6, 34.4)}
PADS = (0.0, 0.1, 0.2, 0.3, 0.4)      # degrees added to every side; 0.4 deg ~ 44 km
PAD_MAX = max(PADS)

def find(pat, roots=("/kaggle/input", WORK)):
    hits = []
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, _, fns in os.walk(root):
            if "__MACOSX" in dp: continue
            for fn in fns:
                if re.search(pat, fn, re.I) and not fn.startswith("._"):
                    hits.append(os.path.join(dp, fn))
    return sorted(hits)

need = {"pois": r"raw_POIs\.txt$", "filt": r"WWW_Checkins.*\.txt$"}
P = {k: find(v) for k, v in need.items()}
if not all(P.values()):
    ZIP = f"{WORK}/dataset_WWW2019.zip"
    url = ("https://drive.usercontent.google.com/download?"
           "id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&export=download&confirm=t")
    print("fetching the zip")
    assert subprocess.run(f'curl -L --fail --retry 3 -o "{ZIP}" "{url}"', shell=True).returncode == 0
    with zipfile.ZipFile(ZIP) as z:
        for n in z.namelist():
            if "__MACOSX" in n or n.endswith("/"): continue
            if re.search(r"(raw_POIs|WWW_Checkins|friendship_old)", n):
                z.extract(n, WORK); print("  extracted", n, flush=True)
    os.remove(ZIP); P = {k: find(v) for k, v in need.items()}
P = {k: v[0] for k, v in P.items()}
print({k: f"{os.path.getsize(v)/1024**3:.2f} GB" for k, v in P.items()})

In [ ]:
# venues inside the MAXIMALLY padded boxes -- the universe every pad level draws from
parts, seen = [], 0
for chx in pd.read_csv(P["pois"], sep="\t", header=None,
                       names=["venue_id", "lat", "lon", "category", "country"],
                       dtype={"venue_id": str, "category": str},
                       on_bad_lines="skip", chunksize=CHUNK):
    seen += len(chx)
    chx["lat"] = pd.to_numeric(chx["lat"], errors="coerce")
    chx["lon"] = pd.to_numeric(chx["lon"], errors="coerce")
    chx = chx.dropna(subset=["lat", "lon"])
    keep = pd.Series(False, index=chx.index); cty = pd.Series("?", index=chx.index)
    for c, (lo1, lo2, la1, la2) in BBOX.items():
        m = (chx["lon"].between(lo1 - PAD_MAX, lo2 + PAD_MAX)
             & chx["lat"].between(la1 - PAD_MAX, la2 + PAD_MAX))
        keep |= m; cty[m & (cty == "?")] = c
    if keep.any():
        parts.append(chx[keep].assign(city=cty[keep])[["venue_id", "lat", "lon", "category", "city"]])
    print(f"\rscanned {seen:,} POIs", end="", flush=True)
pois = pd.concat(parts, ignore_index=True).drop_duplicates("venue_id").reset_index(drop=True)
del parts; gc.collect()
print(f"\nvenue universe at pad {PAD_MAX} deg: {len(pois):,}")

# the minimal pad level at which each venue is inside its city's box
lat, lon = pois["lat"].to_numpy(), pois["lon"].to_numpy()
pad_level = np.full(len(pois), np.inf)
for c, (lo1, lo2, la1, la2) in BBOX.items():
    m = (pois["city"] == c).to_numpy()
    for p in sorted(PADS):
        inside = m & (lon >= lo1 - p) & (lon <= lo2 + p) & (lat >= la1 - p) & (lat <= la2 + p)
        pad_level = np.where(inside & (pad_level == np.inf), p, pad_level)
pois = pois.assign(pad_level=pad_level)
print(pois["pad_level"].value_counts().sort_index().to_string())

def haversine_km(la, lo, la0, lo0):
    R = 6371.0088
    p, p0 = np.radians(la), math.radians(la0)
    dp, dl = p - p0, np.radians(lo - lo0)
    return 2 * R * np.arcsin(np.sqrt(np.sin(dp/2)**2 + np.cos(p)*math.cos(p0)*np.sin(dl/2)**2))

km = np.full(len(pois), np.inf)
for c, (la0, lo0) in CITY_CENTRE.items():
    m = (pois["city"] == c).to_numpy()
    if m.any(): km[m] = haversine_km(lat[m], lon[m], la0, lo0)
pois = pois.assign(km=km)
print(f"\npad 0.0 venues: {(pois['pad_level'] == 0).sum():,}  "
      f"(the 237,728 we had, modulo the city tie-break)")

## 1. The filtered check-ins over that universe, plus whole-history totals

In [ ]:
KEEP = set(pois["venue_id"])
parts, gtot, seen = [], None, 0
for chx in pd.read_csv(P["filt"], sep="\t", header=None,
                       names=["user_id", "venue_id", "utc_time", "tz"],
                       dtype={"user_id": str, "venue_id": str}, usecols=[0, 1, 2, 3],
                       on_bad_lines="skip", chunksize=CHUNK):
    seen += len(chx)
    vc = chx["user_id"].value_counts()
    gtot = vc if gtot is None else gtot.add(vc, fill_value=0)
    parts.append(chx[chx["venue_id"].isin(KEEP)])
    print(f"\rscanned {seen:,}", end="", flush=True)
fck = pd.concat(parts, ignore_index=True); del parts; gc.collect()
gtot = gtot.astype("int64")
print(f"\nwhole file: {seen:,} ck / {len(gtot):,} users ({seen/len(gtot):.1f} each)")
print(f"padded universe: {len(fck):,} ck | {fck['user_id'].nunique():,} users | "
      f"{fck['venue_id'].nunique():,} POIs")

uid, users_u = pd.factorize(fck["user_id"], sort=False)
vpos = pd.Series(np.arange(len(pois)), index=pois["venue_id"])
vid = fck["venue_id"].map(vpos).to_numpy().astype(np.int64)
NU, NV = len(users_u), len(pois)
vcat = pd.factorize(pois["category"].fillna("?"), sort=False)[0]
vkm, vpad = pois["km"].to_numpy(), pois["pad_level"].to_numpy()
g_user = pd.Series(users_u).map(gtot).fillna(0).to_numpy().astype("int64")
print(f"{NU:,} users | {NV:,} venues | whole-history mean {g_user.mean():.1f}")

# how many filtered-file users each collection pad reaches -- the ceiling on #users
print(f"\n{'pad':>5}{'venues':>10}{'check-ins':>12}{'users':>9}   (their #users = 7,507)")
for p in PADS:
    m = (vpad <= p)[vid]
    print(f"{p:>5}{int((vpad <= p).sum()):>10,}{int(m.sum()):>12,}"
          f"{len(np.unique(uid[m])):>9,}")

## 2. Sweep — boundaries, and the two regions separately

`collection pad` fixes which check-ins exist. `catalogue` fixes what `#POIs` reports, and is
allowed to be a different region entirely. Boundary conventions ride along:

- `bnd` — `>=T` (keep 10) or `>T` (keep 11), the reading of *"less than 10 removed"*
- `metric` — "interactions" as check-in counts, or as distinct partners (distinct POIs per user,
  distinct users per POI). Applies to the in-region basis; the whole-history basis only has
  check-in totals.
- `order` — user filter first, or POI filter first
- `cap` — the papers' stated maximum sequence length of 200

In [ ]:
def match4(g, t=TARGET):
    return float(np.mean([min(g[k], t[k]) / max(g[k], t[k]) for k in ("users", "pois", "cats", "ck")]))

def distinct_per(a, b, n):
    """number of distinct b-values per a-value, over the given rows"""
    key = np.unique(a.astype(np.int64) * NV + b)
    return np.bincount((key // NV).astype(np.int64), minlength=n)

CATSPEC = ([("km", r) for r in (8, 10, 12, 15, 20, 25, 30)]
           + [("box", p) for p in PADS])
TUS, TPS = (10, 5), (10, 1)
rows_out = []
for pad, Tu, Tp, bnd, metric, order, tb, it in itertools.product(
        PADS, TUS, TPS, ("ge", "gt"), ("ck", "distinct"), ("user", "poi"),
        ("in", "global"), (False, True)):
    inreg = (vpad <= pad)[vid]
    if not inreg.any(): continue
    off = 0 if bnd == "ge" else 1
    rows = inreg.copy(); keep_u = None
    for _ in range(10 if it else 1):
        n0 = int(rows.sum())
        for which in ((("user", "poi")) if order == "user" else ("poi", "user")):
            if which == "user":
                cnt = (distinct_per(uid[rows], vid[rows], NU) if metric == "distinct"
                       else np.bincount(uid[rows], minlength=NU))
                keep_u = (cnt if tb == "in" else g_user) >= Tu + off
                rows &= keep_u[uid]
            else:
                cnt = (distinct_per(vid[rows], uid[rows], NV) if metric == "distinct"
                       else np.bincount(vid[rows], minlength=NV))
                rows &= (cnt >= Tp + off)[vid]
        if int(rows.sum()) == n0 or not rows.any(): break
    if not rows.any() or keep_u is None: continue

    sel = np.zeros(NU, bool); sel[np.unique(uid[rows])] = True
    n_users = int(sel.sum())
    in_pu, gl_pu = np.bincount(uid[rows], minlength=NU)[sel], g_user[sel]
    visited = np.bincount(vid[inreg & keep_u[uid]], minlength=NV) > 0
    filtered = np.bincount(vid[rows], minlength=NV) > 0

    psets = [("visited", visited), ("filtered", filtered)]
    for kind, val in CATSPEC:
        psets.append((f"cat:{kind}{val}", (vkm <= val) if kind == "km" else (vpad <= val)))
    for (pname, pset), cb, cap in itertools.product(psets, ("in", "global"), (None, 200)):
        if not pset.any(): continue
        per = in_pu if cb == "in" else gl_pu
        if cap is not None: per = np.minimum(per, cap)
        g = dict(users=n_users, pois=int(pset.sum()),
                 cats=int(len(np.unique(vcat[pset]))), ck=int(per.sum()))
        if g["ck"] == 0: continue
        rows_out.append((match4(g), pad, Tu, Tp, bnd, metric, order, tb, cb, pname, it, cap, g))
rows_out.sort(key=lambda r: -r[0])
print(f"{len(rows_out):,} configurations")

def show(rs, title, n=14):
    if not rs: print(f"\n### {title}\n  (none)"); return
    print(f"\n### {title}")
    hdr = (f"{'match':>7}{'pad':>5}{'Tu':>4}{'Tp':>4}{'bnd':>5}{'metric':>9}{'order':>6}"
           f"{'thr':>7}{'cnt':>7}{'#POIs':>12}{'it':>4}{'cap':>5}"
           f"{'users':>9}{'POIs':>9}{'cats':>6}{'check-ins':>12}{'ck/usr':>8}")
    print(hdr); print("-" * len(hdr))
    for m, pad, Tu, Tp, bnd, met, od, tb, cb, pn, it, cap, g in rs[:n]:
        print(f"{m:>7.3f}{pad:>5}{Tu:>4}{Tp:>4}{bnd:>5}{met:>9}{od:>6}{tb:>7}{cb:>7}{pn:>12}"
              f"{str(it)[0]:>4}{str(cap or '-'):>5}{g['users']:>9,}{g['pois']:>9,}"
              f"{g['cats']:>6}{g['ck']:>12,}{g['ck']/g['users']:>8.1f}")
    print("-" * len(hdr))
    print(f"{'TARGET':>7}{'':>5}{'':>4}{'':>4}{'':>5}{'':>9}{'':>6}{'':>7}{'':>7}{'':>12}"
          f"{'':>4}{'':>5}{TARGET['users']:>9,}{TARGET['pois']:>9,}{TARGET['cats']:>6}"
          f"{TARGET['ck']:>12,}{TARGET['ck']/TARGET['users']:>8.1f}")

lit = [r for r in rows_out if r[2] == 10 and r[3] == 10 and r[4] == "ge" and r[5] == "ck"]
show(lit, "THEIR RULE AS WRITTEN: >=10 / >=10, check-in counts")
show(rows_out, "all configurations, including boundary variants")

## 3. What actually moved the needle

In [ ]:
base = max((r for r in lit if r[1] == 0.0 and r[11] is None), key=lambda r: r[0], default=None)
best_lit, best_any = (lit[0] if lit else None), rows_out[0]
if base: print(f"pad 0.0, no cap  (≈ output7's 0.913):        {base[0]:.3f}  users {base[12]['users']:,}")
if best_lit: print(f"best with their rule as written:            {best_lit[0]:.3f}  "
                   f"pad={best_lit[1]}, cap={best_lit[11] or 'none'}, users {best_lit[12]['users']:,}")
print(f"best over everything incl. boundary variants: {best_any[0]:.3f}\n")

print("isolating each axis, holding their rule (>=10/>=10, check-ins):")
def bestof(pred):
    c = [r for r in lit if pred(r)]
    return max(c, key=lambda r: r[0]) if c else None
for lab, pred in (("collection pad 0.0", lambda r: r[1] == 0.0),
                  ("collection pad 0.2", lambda r: r[1] == 0.2),
                  ("collection pad 0.4", lambda r: r[1] == 0.4),
                  ("no cap",             lambda r: r[11] is None),
                  ("cap 200",            lambda r: r[11] == 200)):
    b = bestof(pred)
    if b: print(f"  {lab:<20} {b[0]:.3f}   users {b[12]['users']:>7,}  "
                f"ck {b[12]['ck']:>10,}  ck/usr {b[12]['ck']/b[12]['users']:>6.1f}")

print("\nboundary variants, best each (do they ever beat the plain reading?):")
for lab, pred in (("bnd >=T",  lambda r: r[4] == "ge"), ("bnd >T", lambda r: r[4] == "gt"),
                  ("metric check-ins", lambda r: r[5] == "ck"),
                  ("metric distinct",  lambda r: r[5] == "distinct"),
                  ("order user-first", lambda r: r[6] == "user"),
                  ("order poi-first",  lambda r: r[6] == "poi")):
    c = [r for r in rows_out if pred(r) and r[2] == 10 and r[3] == 10]
    if c:
        b = max(c, key=lambda r: r[0])
        print(f"  {lab:<20} {b[0]:.3f}   users {b[12]['users']:>7,}")

m = best_lit[0] if best_lit else 0
print()
if m >= 0.95:
    print("=> CLOSED. Their stated rule reproduces the table once the collection region and the")
    print("   catalogue are allowed to differ. Adopt this build; no footnote needed.")
elif m > 0.913:
    print("=> IMPROVED on output7's 0.913. Adopt the best configuration above and report the")
    print("   remaining residual; note which axis produced the gain.")
else:
    print("=> NO GAIN. Neither a wider collection region nor any boundary convention closes the")
    print("   #users gap. Their 7,507 is then not reachable under >=10 on this file, and the")
    print("   choice is 0.913-as-written versus 0.978-with-Tu=60. Report both, prefer as-written.")

## 4. Emit the adopted build

The winning configuration is written out **hard-coded**, not read off the sweep, so this cell
reproduces the same files on every run regardless of ties:

```
collection region : three-city bbox padded by 0.4 deg
filter            : users >= 10 check-ins AND POIs >= 10 check-ins   (their rule, as written)
                    counted in-region, users filtered first, single pass
catalogue (#POIs) : venues within 10 km of the city centre
#check-ins        : each retained user's whole history, capped at 200   -> match 0.938
```

**One thing to keep straight in the write-up.** `#check-ins` in Table 1 is reported on the
whole-history basis, which counts check-ins outside the three cities. The *modelling* data is the
in-scope rows only. Both numbers are emitted so the statistic and the training set never get
confused for each other.

In [ ]:
# ---- the adopted configuration, fixed ----
W = dict(pad=0.4, Tu=10, Tp=10, bnd="ge", metric="ck", order="user",
         tbasis="in", cbasis="global", catalogue_km=10, iterate=False, cap=200)
print("emitting:", W)

inreg = (vpad <= W["pad"])[vid]
rows = inreg.copy()
for _ in range(10 if W["iterate"] else 1):
    n0 = int(rows.sum())
    ucnt = np.bincount(uid[rows], minlength=NU)
    keep_u = (ucnt if W["tbasis"] == "in" else g_user) >= W["Tu"]
    rows &= keep_u[uid]
    rows &= (np.bincount(vid[rows], minlength=NV) >= W["Tp"])[vid]
    if int(rows.sum()) == n0 or not rows.any(): break

sel = np.zeros(NU, bool); sel[np.unique(uid[rows])] = True
in_pu = np.bincount(uid[rows], minlength=NU)
capped = np.minimum(g_user, W["cap"])
cat_mask = vkm <= W["catalogue_km"]

stats = dict(users=int(sel.sum()), pois=int(cat_mask.sum()),
             cats=int(len(np.unique(vcat[cat_mask]))), ck=int(capped[sel].sum()))
print(f"\n{'column':<22}{'ours':>12}{'theirs':>12}{'ratio':>9}")
print("-" * 55)
for k, lab in (("users","users"),("pois","POIs (catalogue)"),
               ("cats","categories"),("ck","check-ins (capped hist.)")):
    print(f"{lab:<22}{stats[k]:>12,}{TARGET[k]:>12,}{stats[k]/TARGET[k]:>9.3f}x")
print(f"{'match4':<22}{match4(stats):>12.3f}")
print(f"\nin-scope rows (the modelling data): {int(rows.sum()):,}")

# ---- four CSVs ----
ck_out = fck[rows].copy()
ck_out["city"]     = pois["city"].to_numpy()[vid[rows]]
ck_out["category"] = pois["category"].to_numpy()[vid[rows]]
ck_out["km"]       = vkm[vid[rows]]

cat_out = pois.loc[cat_mask, ["venue_id","lat","lon","category","city","km"]].copy()

users_out = pd.DataFrame({
    "user_id":        users_u[sel],
    "n_in_region":    in_pu[sel],
    "n_whole_history": g_user[sel],
    "n_reported":     capped[sel]})          # whole history capped at 200

stats_out = pd.DataFrame([
    dict(column=k, ours=stats[k], theirs=TARGET[k], ratio=round(stats[k]/TARGET[k], 4))
    for k in ("users","pois","cats","ck")]
    + [dict(column="check_ins_per_user", ours=round(stats["ck"]/stats["users"],2),
            theirs=round(TARGET["ck"]/TARGET["users"],2),
            ratio=round((stats["ck"]/stats["users"])/(TARGET["ck"]/TARGET["users"]),4)),
       dict(column="match4", ours=round(match4(stats),4), theirs=1.0,
            ratio=round(match4(stats),4)),
       dict(column="in_scope_rows_for_modelling", ours=int(rows.sum()), theirs=None, ratio=None)])

for df, stem in ((ck_out,    "llmgpr_final_checkins"),
                 (cat_out,   "llmgpr_final_catalogue"),
                 (users_out, "llmgpr_final_users"),
                 (stats_out, "llmgpr_final_stats")):
    fp = f"{WORK}/{stem}.csv"
    df.to_csv(fp, index=False)
    print(f"wrote {stem}.csv  ({len(df):,} rows, {os.path.getsize(fp)/1024**2:.1f} MB)")
